# Logical Failure Investigation

This notebook investigates the behaviour of the current logical-failure
heuristic used by the MWPM decoder.

Key observations:

- Logical failure rates remain near 20% even when p1=0 and ro=0.
- Failure rates show little dependence on physical error rate.
- Certain ancilla-to-boundary matchings are classified as logical failures.

These results suggest that the current correction-span heuristic does not
correctly identify logical operators.

## Test 1 
Expected: No syndrome defect -> No matching pair -> No logical failure

In [1]:
from qec.decoder import mwpm_pairs, correction_spans_code

pairs = mwpm_pairs([], "vertical", distance=3)

print("pairs =", pairs)

print(
    correction_spans_code(
        pairs,
        boundary_mode="vertical",
        distance=3,
    )
)

pairs = []
False


## Test 2
Expected: Matched to boundary -> Should not automatically imply logical failure

In [2]:
pairs = mwpm_pairs(
    [0],
    "vertical",
    distance=3,
)

print(pairs)

print(
    correction_spans_code(
        pairs,
        "vertical",
        3,
    )
)

[('B', 'a0')]
False


## Test 3 
Expected: Short correction chain -> No logical failure

In [3]:
pairs = mwpm_pairs(
    [0, 1],
    "vertical",
    distance=3,
)

print(pairs)

print(
    correction_spans_code(
        pairs,
        "vertical",
        3,
    )
)

[('a1', 'a0')]
False


## Test 4: Long Correction Chain

This test investigates whether the current heuristic incorrectly
classifies long correction chains as logical failures.

The pair ("a0", "a2") spans a large fraction of the code in the
vertical direction, but does not necessarily correspond to a
logical operator.

In [4]:
pairs = [("a0", "a2")]
a0 = (0.5, 0.5)
a2 = (1.5, 0.5)

print(
    correction_spans_code(
        pairs,
        "vertical",
        3,
    )
)

from qec.geometry import generate_ancilla_positions

generate_ancilla_positions(3)

False


{0: (0.5, 0.5), 1: (0.5, 1.5), 2: (1.5, 0.5), 3: (1.5, 1.5)}

In [9]:
for pairs in [
    [("a0", "B")],
    [("a2", "B")],
]:
    print(pairs)

    print(
        correction_spans_code(
            pairs,
            "vertical",
            3,
        )
    )

[('a0', 'B')]
False
[('a2', 'B')]
True


In [6]:
from qec.geometry import generate_ancilla_positions

generate_ancilla_positions(5)

pairs = [("a0", "a12")]

correction_spans_code(
    pairs,
    "vertical",
    5,
)

False

In [7]:
from qec.simulation import logical_failure_rates_single

fx, fz = logical_failure_rates_single(
    distance=3,
    shots=1000,
    p1=0.0,
    ro=0.0,
)

print(fx, fz)

0.197 0.181


In [8]:
for p in [0.001, 0.005, 0.01]:
    fx, fz = logical_failure_rates_single(
        distance=3,
        shots=1000,
        p1=p,
        ro=0.0,
    )

    print(p, fx, fz)

0.001 0.212 0.204
0.005 0.194 0.196
0.01 0.193 0.189


In [10]:
from qec.geometry import (
    generate_ancilla_positions,
    code_boundaries,
)

anc_pos = generate_ancilla_positions(3)
bounds = code_boundaries(3)

print("ancilla positions:")
print(anc_pos)

print("\nboundaries:")
print(bounds)

ancilla positions:
{0: (0.5, 0.5), 1: (0.5, 1.5), 2: (1.5, 0.5), 3: (1.5, 1.5)}

boundaries:
{'top': -0.5, 'bottom': 2.5, 'left': -0.5, 'right': 2.5, 'span': 3.0}


In [11]:
for pairs in [
    [("a0", "a3")],
    [("a1", "a2")],
    [("a0", "a2")],
    [("a1", "a3")],
]:
    print(
        pairs,
        correction_spans_code(
            pairs,
            "vertical",
            3,
        )
    )

[('a0', 'a3')] False
[('a1', 'a2')] False
[('a0', 'a2')] False
[('a1', 'a3')] False
